# 机器学习应用开发实战 - 作业

任务：
1. 去掉表中的第一列
2. 补充缺失值
3. PCA降维成3维
4. K-Means聚类成两簇

## 1. 导入所需库

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 中文显示支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 2. 读取数据

> 请将下方文件路径替换为你自己的数据文件路径（支持 .csv / .xlsx）

In [ ]:
# ====== 修改这里 ======
df = pd.read_csv('your_data.csv')  # 替换为你的文件路径
# 如果是Excel文件，用下面这行：
# df = pd.read_excel('your_data.xlsx')
# ======================

print('原始数据形状:', df.shape)
df.head(10)

## 步骤1：去掉表中的第一列

In [ ]:
# 删除第一列（按位置索引，不依赖列名）
first_col_name = df.columns[0]
df = df.drop(columns=[first_col_name])

print(f'已删除第一列: "{first_col_name}"')
print('删除后形状:', df.shape)
df.head(10)

## 步骤2：补充缺失值

In [ ]:
# 查看缺失值情况
print('各列缺失值数量:')
print(df.isnull().sum())
print(f'\n总缺失值: {df.isnull().sum().sum()}')

In [ ]:
# 对数值列用均值填充，对非数值列用众数填充
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

if num_cols:
    imputer_num = SimpleImputer(strategy='mean')
    df[num_cols] = imputer_num.fit_transform(df[num_cols])
    print(f'数值列({len(num_cols)}列)已用均值填充')

if cat_cols:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
    print(f'分类列({len(cat_cols)}列)已用众数填充')

print(f'\n填充后剩余缺失值: {df.isnull().sum().sum()}')
df.head(10)

## 数据预处理：标准化

PCA对数据尺度敏感，聚类前需要标准化

In [ ]:
# 若存在非数值列，进行One-Hot编码
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
    print(f'One-Hot编码后形状: {df.shape}')

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f'标准化后数据形状: {X_scaled.shape}')

## 步骤3：PCA降维成3维

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print(f'PCA降维后形状: {X_pca.shape}')
print(f'各主成分方差解释比例: {pca.explained_variance_ratio_}')
print(f'累计方差解释率: {pca.explained_variance_ratio_.sum():.4f}')

# 查看降维结果
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])
pca_df.head(10)

## 步骤4：K-Means聚类成两簇

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

pca_df['cluster'] = labels

print(f'聚类标签分布:')
print(pca_df['cluster'].value_counts().sort_index())
print(f'\n聚类中心（PCA空间）:')
print(kmeans.cluster_centers_)
pca_df.head(10)

## 可视化

In [ ]:
fig = plt.figure(figsize=(16, 6))

# 3D散点图
ax1 = fig.add_subplot(131, projection='3d')
scatter = ax1.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
                       c=labels, cmap='Set1', alpha=0.6, edgecolors='k', linewidth=0.5)
ax1.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            kmeans.cluster_centers_[:, 2],
            c='black', marker='X', s=200, label='聚类中心')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_zlabel('PC3')
ax1.set_title('PCA降维 + K-Means聚类 (3D)')
ax1.legend()

# PC1 vs PC2
ax2 = fig.add_subplot(132)
ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='Set1', alpha=0.6, edgecolors='k', linewidth=0.5)
ax2.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='black', marker='X', s=200)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_title('PC1 vs PC2')

# PC1 vs PC3
ax3 = fig.add_subplot(133)
ax3.scatter(X_pca[:, 0], X_pca[:, 2], c=labels, cmap='Set1', alpha=0.6, edgecolors='k', linewidth=0.5)
ax3.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 2],
            c='black', marker='X', s=200)
ax3.set_xlabel('PC1')
ax3.set_ylabel('PC3')
ax3.set_title('PC1 vs PC3')

plt.tight_layout()
plt.savefig('pca_kmeans_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('图表已保存为 pca_kmeans_result.png')